# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
from google import genai

In [4]:
# Initialize and constants

load_dotenv(override=True)
# api_key = os.getenv('OPENAI_API_KEY')

# if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
#     print("API key looks good so far")
# else:
#     print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# MODEL = 'gpt-5-nano'
# openai = OpenAI()
client = genai.Client(vertexai=True, 
project=os.getenv("GOOGLE_CLOUD_PROJECT"), 
location=os.getenv("GOOGLE_CLOUD_LOCATION"))

In [6]:
links = fetch_website_links("https://edwarddonner.com")
#links = fetch_website_links("https://grainzy.co.in")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [19]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [17]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [18]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [38]:
from google.genai import types
MODEL = "gemini-2.5-flash"
def select_relevant_links(url):
    # response = openai.chat.completions.create(
    #     model=MODEL,
    #     messages=[
    #         {"role": "system", "content": link_system_prompt},
    #         {"role": "user", "content": get_links_user_prompt(url)}
    #     ],
    #     response_format={"type": "json_object"}
    # )
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    messages = link_system_prompt + get_links_user_prompt(url)
    response = client.models.generate_content(model=MODEL,
    contents=messages,
    config=types.GenerateContentConfig(
        response_mime_type="application/json"
    )
    )
    result = response.text
    #print(result)
    links = json.loads(result)
    return links
    

In [39]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-2.5-flash


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [40]:
# def select_relevant_links(url):
#     print(f"Selecting relevant links for {url} by calling {MODEL}")
#     response = openai.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": link_system_prompt},
#             {"role": "user", "content": get_links_user_prompt(url)}
#         ],
#         response_format={"type": "json_object"}
#     )
#     result = response.choices[0].message.content
#     links = json.loads(result)
#     print(f"Found {len(links['links'])} relevant links")
#     return links

In [41]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-2.5-flash


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'related company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [42]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'enterprise solutions page',
   'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'learning resources page', 'url': 'https://huggingface.co/learn'},
  {'type': 'documentation page', 'url': 'https://huggingface.co/docs'},
  {'type': 'brand guidelines page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://huggingface.co/join'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'github profile', 'url': 'https://github.com/huggingface'},
  {'type': 'discord invitation', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'social media page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'social media page',

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [43]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [44]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
meta-models/Muse-Glimmer-30B
Updated
2 days ago
•
121k
•
1.42k
MiniMaxAI/MiniMax-H3
Updated
1 day ago
•
1.61M
•
3.83k
Qwen/Qwen3.8-2.4T-A95B
Updated
1 day ago
•
1.01k
•
789
Lightricks/LTX-2.5
Updated
1 day ago
•
57.3k
•
724
deepseek-ai/DeepSeek-V4-Flash-0731
Updated
13 days ago
•
1.43M
•
3.32k
Browse 2

In [54]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [46]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [47]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nmeta-models/Muse-Glimmer-30B\nUpdated\n2 days ago\n•\n121k\n•\n1.42k\nMiniMaxAI/MiniMax-H3\nUpdated\n1 day ago\n•\n1

In [50]:
def create_brochure(company_name, url):
    # response = openai.chat.completions.create(
    #     model="gpt-4.1-mini",
    #     messages=[
    #         {"role": "system", "content": brochure_system_prompt},
    #         {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
    #     ],
    # )
    # result = response.choices[0].message.content
    #print(f"Selecting relevant links for {url} by calling {MODEL}")
    messages = brochure_system_prompt + get_brochure_user_prompt(company_name, url)
    response = client.models.generate_content(model=MODEL,
    contents=messages#,
    # config=types.GenerateContentConfig(
    #     response_mime_type="application/json"
    # )
    )
    result = response.text
    #print(result)
    #links = json.loads(result)
    display(Markdown(result))

In [51]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Hugging Face: The AI Community Building the Future

Hugging Face is the premier platform where the global machine learning community collaborates to create, discover, and deploy the future of artificial intelligence. We are the "Home of Machine Learning," providing essential tools and infrastructure for everyone from individual developers to large enterprises. Our mission is to democratize AI by fostering an open, collaborative ecosystem.

**For Prospective Customers & Developers**

Unlock unparalleled innovation with Hugging Face's comprehensive ecosystem:

*   **Models:** Access and contribute to over 2 million cutting-edge machine learning models for diverse tasks, including trending large language models and specialized applications.
*   **Datasets:** Fuel your AI projects with a vast collection of over 500,000 datasets, ensuring robust training and evaluation.
*   **Spaces:** Bring your AI applications to life instantly. Explore over 1 million interactive demos or effortlessly deploy your own with our user-friendly interface.
*   **Enterprise Solutions:** For organizations requiring enhanced capabilities, our Enterprise offerings, Hugging Face PRO, Inference Providers, Inference Endpoints, and Storage Buckets deliver scalable, secure, and fully supported AI infrastructure tailored to your business needs.

**For Investors**

Invest in the foundational infrastructure of the AI revolution. Hugging Face stands at the epicenter of machine learning innovation, driving the industry forward with an ever-expanding community and critical tools. Our platform, hosting millions of models, datasets, and applications, is indispensable for AI development, presenting significant growth opportunities in a rapidly evolving market. We are building the backbone for the next generation of intelligent systems.

**For Recruits & Community Members**

Join a dynamic and collaborative community that is actively shaping the future of AI. At Hugging Face, our culture thrives on open source principles, collective intelligence, and a shared passion for pushing technological boundaries.

*   **Company Culture:** We are fundamentally a community – "the AI community building the future." Our environment champions collaboration, open source contributions, and continuous learning. Engage with peers and experts through our active Discord, forums, and GitHub, contributing to projects that impact the entire ML ecosystem.
*   **Career & Impact:** While specific job listings are not provided here, our rapid growth and extensive offerings across models, datasets, and applications mean diverse and impactful opportunities for those passionate about advancing AI. Join a team that's at the forefront of machine learning, making AI accessible and powerful for everyone.

**Discover. Create. Collaborate.**

Explore the vast resources at Hugging Face, join our vibrant community, and become an integral part of shaping the future of artificial intelligence.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [52]:
def stream_brochure(company_name, url):
    # stream = openai.chat.completions.create(
    #     model="gpt-4.1-mini",
    #     messages=[
    #         {"role": "system", "content": brochure_system_prompt},
    #         {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
    #       ],
    #     stream=True
    # ) 
    messages = brochure_system_prompt + get_brochure_user_prompt(company_name, url)
    stream = client.models.generate_content_stream(model=MODEL,
    contents=messages
    # config=types.GenerateContentConfig(
    #     response_mime_type="application/json"
    # )
    )   
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.text or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [53]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash


**Hugging Face: The AI Community Building the Future**

Welcome to Hugging Face, the leading platform where the global machine learning community collaborates to create and deploy the future of artificial intelligence. We are dedicated to democratizing AI by providing open tools, models, and datasets, fostering an environment of shared knowledge and innovation.

**What We Offer:**

Hugging Face provides a comprehensive ecosystem for all your ML needs:

*   **Models:** Access and contribute to over 2 million pre-trained models for a vast array of AI tasks.
*   **Datasets:** Discover and utilize over 500,000 datasets to train and refine your machine learning solutions.
*   **Spaces:** Easily deploy, showcase, and explore over 1 million ML applications and interactive demos.
*   **Enterprise Solutions:** For businesses and teams, we offer robust "Team & Enterprise Plans," "Hugging Face PRO," dedicated "Enterprise Support," and scalable "Inference Providers" and "Storage Buckets" to power secure and efficient AI development and deployment.

**Our Culture & Community:**

At the heart of Hugging Face is an vibrant, open-source-driven community. We champion collaboration, innovation, and ethical AI development, believing that collective intelligence accelerates progress. Our platform serves as "The Home of Machine Learning," where researchers, developers, and practitioners worldwide connect, share, and build together.

**Who We Serve:**

*   **Prospective Customers:** Whether you're an individual exploring AI, a research team pushing boundaries, or an enterprise scaling ML operations, Hugging Face offers the foundational tools and infrastructure to succeed.
*   **Investors:** Join us in empowering the next generation of AI innovation. Invest in a rapidly growing platform at the core of the global machine learning ecosystem, driven by an engaged community and increasing enterprise adoption.
*   **Recruits:** Contribute to shaping the AI landscape alongside a dynamic and impactful team. We seek passionate individuals dedicated to open collaboration, community impact, and advancing machine learning for a better future. A company at the forefront of AI constantly looks for talent to join its mission.

Join the movement and build the future of AI with Hugging Face.

In [55]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash


**HuggingFace: Where AI Gets Its Hugs (And Builds the Future)**

Tired of your AI dreams feeling like a lonely algorithm in a vast, cold data center? Welcome to HuggingFace, the warmest, fuzziest, and most collaborative corner of the machine learning universe! We're not just a company; we're **the AI community building the future**, one model, dataset, and AI app at a time.

---

### What We Do: The Ultimate AI Potluck!

Imagine a place where over **2 million models** are casually hanging out, **500,000+ datasets** are ready to be devoured, and **1 million applications** are just begging to be explored. That's us! Whether you're looking for the next big thing in video generation (yes, we have that!) or a free AI detector (also that!), we've got your neural network's every whim covered. We're the platform where groundbreaking AI goes from "brilliant idea" to "bizarrely effective demo" faster than you can say "large language model."

---

### Who Should Get a Hug From Us?

*   **For Prospective Customers (You, the AI Innovators!):** Stop reinventing the wheel! Dive into a collaborative ecosystem with enterprise support, inference providers, and all the tools you need to build, deploy, and scale your AI solutions. We'll help you make your AI smarter, faster, and probably a lot more charming.

*   **For Savvy Investors (You, with a keen eye for the next big thing!):** Want to invest in the future? We *are* the future! HuggingFace is at the epicenter of AI innovation, driving collaboration and democratizing access to cutting-edge ML. Our growth trajectory is steeper than a gradient descent on a roller coaster – a good roller coaster, we promise!

*   **For Future Team Members (You, who crave a career with impact and actual humans!):** Our **company culture** is all about **community, collaboration, and open-source love**. We believe the best AI is built together, in a vibrant space where ideas are shared, challenged, and celebrated. If you're a brilliant mind seeking to push boundaries, learn from the best, and make a real impact (without being hugged *too* tight by a robot), then your next **career** move is calling. Come build the future with us!

---

So, whether you want to **model** the next big breakthrough, **dataset** your way to discovery, or find your **space** in the exciting world of AI, come join HuggingFace. We're open-source, open-minded, and always ready for a good collaboration.

**Don't just observe the AI revolution – *hug* it!**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>